# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive full credit on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**For half credit, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description and feel free to experiment as you see fit.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [ ]:
# PUT YOUR IMPORTS HERE
import polars as pl
import altair as alt
from census import Census

In [ ]:
def legislators_df():
    """This function reads in the csv, creates the state column and returns it"""

    read_df = pl.read_csv("data/legislators.csv")
    state_df = read_df.with_columns(
        pl.col("jurisdiction")
        .str.extract(r"(?:state:|district:|territory:)(\S*)\/")
        .str.to_uppercase()
        .alias("state")
    )

    return state_df


# render dataframe
legislators_df()

### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [ ]:
def states_by_gender_initial(df):
    """This function gets the data in state and gender level and plots it"""
    df = df.group_by("state", "gender").len()

    bar_chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            alt.X("state:N").title("State"),
            alt.Y("len:Q").title("Count of Legislators"),
            alt.Color("gender:N").title("Gender"),
        )
    )

    return bar_chart


# render chart
states_by_gender_initial(legislators_df())

### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [ ]:
def states_by_gender_improved(df):

    df_percent = (
        df.group_by("state", "gender")
        .len()
        .with_columns(
            percent_legislators=(pl.col("len") / pl.col("len").sum()).over("state")
        )
        .sort("state", "gender")
    )

    base = alt.Chart(df_percent)

    bar_chart = base.mark_bar().encode(
        alt.X("state:N").title("State"),
        alt.Y("percent_legislators:Q")
        .title("Percent of Legislators")
        .axis(format="%")
        .scale(domain=[0, 1]),
        alt.Color("gender:N")
        .title("Gender")
        .scale(
            domain=["Female", "Male", "Other"], range=["#8700F9", "#00C4A9", "#733A04"]
        ),
    )

    rule = base.mark_rule(strokeDash=[2, 2], color="red").encode(y=alt.datum(0.5))

    return bar_chart + rule


# Resources referenced:
# https://stackoverflow.com/questions/78571071/altair-show-axis-tick-labels-as-percentage-point
# https://stackoverflow.com/questions/62281179/how-to-adjust-scale-ranges-in-altair

# render chart
states_by_gender_improved(legislators_df())

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [ ]:
def party_control_raw(df):

    df_percent = (
        df.group_by("state", "party")
        .len()
        .with_columns(
            percent_legislators=(pl.col("len") / pl.col("len").sum()).over("state")
        )
        .sort("state", "party")
    )

    base = alt.Chart(df_percent)

    bar_chart = base.mark_bar().encode(
        alt.X("state:N").title("State"),
        alt.Y("percent_legislators:Q")
        .title("Percent of Legislators")
        .axis(format="%")
        .scale(domain=[0, 1]),
        alt.Color("party:N").title("Party"),
    )

    rule = base.mark_rule(strokeDash=[2, 2], color="red").encode(y=alt.datum(0.5))

    return bar_chart + rule


party_control_raw(legislators_df())

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [ ]:
def clean_party_df():
    # start with the DataFrame from part 1 & return transformed copy
    df = legislators_df()
    df_clean = df.with_columns(
        pl.when(pl.col("party").str.to_lowercase().str.contains("democratic"))
        .then(pl.lit("D"))
        .when(pl.col("party").str.to_lowercase().str.contains("republican"))
        .then(pl.lit("R"))
        .otherwise(pl.lit("O"))
        .alias("party_code")
    ).filter(~pl.col("state").is_in(["DC", "NE", "PR"]))

    return df_clean


clean_party_df()

### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [ ]:
def party_control_by_chamber(df):

    df_percent = (
        df.group_by("state", "party_code", "type")
        .len()
        .with_columns(
            percent_legislators=(pl.col("len") / pl.col("len").sum()).over(
                ["state", "type"]
            )
        )
        .sort("state", "party_code")
    )

    base = alt.Chart(df_percent)

    bar_chart = base.mark_bar().encode(
        alt.X("percent_legislators:Q")
        .title("Percent Legislators")
        .axis(format="%")
        .scale(domain=[0, 1]),
        alt.Y("state:N").title("State"),
        alt.Color("party_code:N")
        .scale(domain=["D", "O", "R"], range=["lightblue", "lightgreen", "pink"])
        .title("Party Code"),
    )

    rule = base.mark_rule().encode(x=alt.datum(0.5))

    bar_rule = bar_chart + rule
    faceted = bar_rule.facet(column="type:N")

    return faceted


party_control_by_chamber(clean_party_df())

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [ ]:
def population_combined_df():

    pop_df = pl.read_csv("data/populations.csv")
    leg_df = legislators_df().with_columns(pl.col("state").str.to_lowercase())
    merged_df = leg_df.join(other=pop_df, left_on="state", right_on="state")
    return merged_df


population_combined_df()

### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [ ]:
def scatter_pop_size():
    df = population_combined_df()
    df_by_state = df.group_by("state", "pop_2020", "type").len()

    base_lower = alt.Chart(df_by_state.filter(pl.col("type") == "lower"))
    base_upper = alt.Chart(df_by_state.filter(pl.col("type") == "upper"))

    scatter_lower = base_lower.mark_point(shape="triangle-down", color="orange").encode(
        alt.X("pop_2020").title("Population 2020"),
        alt.Y("len").title("Number of Seats"),
    )

    scatter_upper = base_upper.mark_point(
        shape="triangle-up", color="lightblue"
    ).encode(alt.X("pop_2020"), alt.Y("len"))

    return scatter_lower + scatter_upper


scatter_pop_size()

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [ ]:
def scatter_pop_size_regression():
    df = population_combined_df()
    df_by_state = df.group_by("state", "pop_2020", "type").len()

    base_lower = alt.Chart(df_by_state.filter(pl.col("type") == "lower"))
    base_upper = alt.Chart(df_by_state.filter(pl.col("type") == "upper"))

    scatter_lower = base_lower.mark_point(shape="triangle-down", color="orange").encode(
        alt.X("pop_2020").title("Population 2020"),
        alt.Y("len").title("Number of Seats"),
    )

    scatter_upper = base_upper.mark_point(shape="triangle-up", color="purple").encode(
        alt.X("pop_2020"), alt.Y("len")
    )

    return (
        scatter_lower
        + scatter_lower.transform_regression("pop_2020", "len").mark_line(
            color="orange"
        )
        + scatter_upper
        + scatter_upper.transform_regression("pop_2020", "len").mark_line(
            color="purple"
        )
    )


scatter_pop_size_regression()

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [ ]:
def actions_df():
    df_read = pl.read_csv("data/actions_il-in-mi-wi_2021-2024.csv")
    df_date = df_read.with_columns(pl.col("date").str.slice(0, 10)).with_columns(
        pl.col("date").str.to_date("%Y-%m-%d")
    )
    return df_date


actions_df()

### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [ ]:
def actions_heatmap_scaled(df):
    df_grouped = df.group_by("state", "date").len()

    base = alt.Chart(df_grouped)

    heatmap = base.mark_rect().encode(
        alt.X("yearweek(date):O"), alt.Y("state:O"), alt.Color("len:Q")
    )
    return heatmap


actions_heatmap_scaled(actions_df())

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [ ]:
# MODIFY THIS LINE (exclude IL)
actions_heatmap_scaled(actions_df().filter(pl.col("state") != "IL"))

In [ ]:
# MODIFY THIS LINE (only IL)
actions_heatmap_scaled(actions_df().filter(pl.col("state") == "IL"))

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [ ]:
def actions_cumulative(df):

    df = df.sort(["state", "date"])
    base = alt.Chart(df)

    # Disable max-row error because we have upwards of 2000 rows
    alt.data_transformers.disable_max_rows()

    chart = (
        base.transform_aggregate(count="count(*)", groupby=["state", "date"])
        .transform_window(
            frame=[None, 0],
            total_seats="sum(count)",
            groupby=["state"],
        )
        .mark_line()
        .encode(
            alt.X("yearmonthdate(date)").title("Date"),
            alt.Y("total_seats:Q").title("Cumulative Actions To Date"),
            alt.Color("state:N").title("State"),
        )
        .properties(width=600)
    )

    return chart


actions_cumulative(actions_df())

## Part 5. Replicating a Chart

Find a chart (perhaps the one you're using for your first Analysis post!) and do your best to replicate it with Altair. Since you likely won't have access to the raw data-- start by mocking up a dataset that is similar enough to what your chart uses, don't worry about exact values, just that the correct attributes are present with some variation.

Your replicated chart should use similar colors, labels, etc. to the one that you chose to reproduce. Again, we aren't looking for a pixel-perfect replica, but something that pushes you to explore parts of Altair's documentation/features that you didn't use above.


I will be trying to make a radial chart to convey the visualization in this [article](https://www.nytimes.com/interactive/2026/07/01/us/america-identity-ancestry-census.html) that shows the number of immigrants in the US over the years and their country of origin. I use the ACS data on country of origin which was also used by the author.

In [ ]:
api_key = "my_api_key" # please add your personal API key - I removed mine after fetching the data

census = Census(api_key, year=2023)

census_columns = {
    "B05006_002E": "Europe",
    "B05006_047E": "Asia",
    "B05006_095E": "Africa",
    "B05006_130E": "Oceania",
    "B05006_138E": "Americas",
}

data = census.acs5.get(
    ["NAME"] + list(census_columns.keys()),
    geo={"for": "state:*"},
)

country_of_origin_df = pl.DataFrame(data).rename(census_columns)

In [ ]:
country_of_origin_df

In [ ]:
long_df = country_of_origin_df.select(
    pl.sum("Europe", "Asia", "Africa", "Oceania", "Americas")
    ).unpivot(value_name="count", variable_name="origin").with_columns(
    pl.col("count").cast(pl.Int64)
    )

In [ ]:
long_df

In [ ]:
base = alt.Chart(long_df).encode(
    alt.Theta("count:Q").stack(True),
    alt.Radius("count:Q").scale(type="log", zero=True, rangeMin=20),
    alt.Color("origin:N").title("Immigrant's Country of Origin")
)
c1 = base.mark_arc(innerRadius=30, stroke="#fff", opacity=0.7)
c2 = base.mark_text(radiusOffset=10, fontSize=16, color="black").encode(text="count:Q")

c1 + c2